# Memory Management

Context-window economics, compaction, and long-term memory retrieval.

This notebook is an original tutorial (rewritten for depth, 2026 practice).

## Learning Objectives

- Distinguish short-term (in-context) from long-term (external store) memory, and semantic/episodic/procedural types.
- Manage the context window as a budget: what to keep verbatim, what to compact, what to evict.
- Implement summarization/compaction of conversation history with preservation rules.
- Explain retrieval-based memory (embedding search over past facts) vs stuffing everything in context.

## Mental Model

Memory is **context-window economics**. The window is a scarce, expensive
buffer; everything else is a cheaper, slower store you page from. Three
moves, exactly like an OS memory hierarchy:

- **Keep** (in window): system prompt, recent turns, active task state.
- **Compact** (summarize in place): old conversation → a dense summary that
  preserves decisions, constraints, and open questions.
- **Externalize + retrieve**: durable facts go to a store (relational,
  vector, or files); retrieved back in only when relevant.

The failure mode of naive agents is treating the window as infinite until it
isn't — quality degrades long before hard truncation (long-context recall
falls with length; see notebook 26's lost-in-the-middle numbers).

## Important Concepts

- **Short-term memory** = the context window itself. **Long-term memory** = anything that survives the session: DB rows, vector store, memory files.
- **Semantic** (facts about the user/world), **episodic** (what happened in past sessions), **procedural** (how to do things — learned preferences, house rules). Production memory systems store all three with different schemas.
- **Compaction**: when the window approaches a threshold (e.g. 70-80%), replace the oldest turns with an LLM-written summary. Preservation rules matter more than the summarizer: keep decisions, constraints, unresolved questions, and exact identifiers verbatim.
- **Retrieval-based memory**: embed memory entries; at each turn, retrieve top-k relevant to the current query and inject. Scales to unbounded history at constant context cost.
- **Write policies**: what earns a memory? Explicit user asks, repeated corrections, stable preferences — not every utterance. Bad write policies are why "agent memory" products feel creepy or noisy.
- **Frameworks (one line)**: LangGraph checkpointer = thread state (short-term across restarts) + store API (long-term); file-based memory (CLAUDE.md-style) = plain-text procedural memory read each session.

## Need-To-Know Coverage Checklist

- [x] Short-term vs long-term; semantic/episodic/procedural with examples.
- [x] Context budget management: keep / compact / externalize.
- [x] Compaction thresholds and preservation rules, in code.
- [x] Retrieval-based memory (embedding top-k) vs context stuffing.
- [x] Memory write policies and staleness/correction handling.
- [x] Privacy: memory is PII; scoping, TTLs, user deletion.

## Deep Study Notes

### The budget loop

Every turn: estimate tokens (system + memories + history + new input +
headroom for output). If over threshold, compact before calling the model —
not after a context-length error. Reserve headroom: an agent that fills 100%
of its window can't think.

### Compaction that doesn't lose the plot

An LLM summary of 30 turns is lossy by design; the craft is *what must
survive verbatim*:
- decisions made ("we chose Postgres") and their reasons,
- constraints ("must ship Friday", "never email the client directly"),
- open questions and current task state,
- exact identifiers (ids, file paths, numbers) — summaries mangle these.

Structure the compact as fields, not prose. Recompacting a compact compounds
loss — prefer compacting fresh turns into the existing structured summary.

### Retrieval-based memory

Store entries as (text, embedding, type, timestamps, source). At each turn:
embed the query, retrieve top-k by cosine similarity (see notebook 25 for
mechanics), inject as a labeled block. Two production details:
- **Recency + relevance**: pure similarity retrieves stale facts; score =
  similarity x recency decay, or filter superseded entries.
- **Corrections**: when the user contradicts a stored memory, *update or
  tombstone the old entry* — the classic bug is retrieving both "prefers
  formal tone" and its correction and letting the model pick.

### Privacy

Memory is PII by construction. Scope per user/tenant, TTL what's not
load-bearing, support deletion, and never let one user's memories leak into
another's retrieval — a real incident class in memory-enabled products.

## Common Failure Modes

- Compacting only on context-length errors → the model already degraded turns ago.
- Prose summaries losing identifiers and constraints → the agent re-asks or violates them.
- Retrieving contradictory memories (fact + its correction) → nondeterministic behavior.
- Writing everything to memory → noisy retrieval, creepy product, PII liability.
- Recompacting compacts repeatedly → compounding loss of decisions.
- One global memory store across users → cross-tenant leakage.

## Implementation Notes

- Track token counts per section (system/memories/history); log them — context bloat is diagnosable only if measured.
- Compact into a structured schema (decisions/constraints/open_questions/state), not free prose.
- Memory writes go through an explicit policy function; log why each entry was written.
- Version memory entries; corrections tombstone their predecessors.

In [ ]:
"""Context-budget management with structured compaction, plus a
recency-aware memory retriever with correction handling.
"""

# --- 1. compaction under a token budget -----------------------------------
BUDGET = 100  # toy token budget (prod: model window minus output headroom)
est = lambda text: len(text.split())  # toy tokenizer


def compact(turns):
    """Structured compaction: preserve decisions/constraints/ids verbatim."""
    summary = {"decisions": [], "constraints": [], "state": ""}
    for t in turns:
        if "decided" in t:
            summary["decisions"].append(t)          # verbatim, not paraphrased
        elif "must" in t or "never" in t:
            summary["constraints"].append(t)
    summary["state"] = f"({len(turns)} earlier turns compacted)"
    return summary


history = [
    "user: we decided to use Postgres for the transcript store",
    "assistant: noted, setting that up",
    "user: the demo must ship Friday",
    "assistant: understood",
    "user: how should we shard the vector index?",
    "assistant: for 250k vectors pgvector needs no sharding at all",
    "user: ok what about the dashboard queries",
]

used = sum(est(t) for t in history)
if used > BUDGET * 0.7:  # compact at 70%, not at overflow
    keep_recent = history[-2:]
    summary = compact(history[:-2])
    print(f"compacted: {used} -> ~{est(str(summary)) + sum(est(t) for t in keep_recent)} tokens")
    print(" preserved decisions:", summary["decisions"])
    print(" preserved constraints:", summary["constraints"])

# --- 2. retrieval-based memory with corrections ----------------------------
MEMORIES = [
    {"id": 1, "text": "user prefers formal tone in emails", "age_days": 90, "superseded_by": 3},
    {"id": 2, "text": "user's company is AcmeCo, fiscal year ends March", "age_days": 40, "superseded_by": None},
    {"id": 3, "text": "user now prefers casual tone in emails", "age_days": 5, "superseded_by": None},
]


def retrieve(query_terms, k=2):
    """similarity x recency, excluding tombstoned entries."""
    live = [m for m in MEMORIES if m["superseded_by"] is None]
    def score(m):
        sim = len(set(query_terms) & set(m["text"].split())) / len(query_terms)
        recency = 1.0 / (1 + m["age_days"] / 30)
        return sim * recency
    return sorted(live, key=score, reverse=True)[:k]


hits = retrieve(["tone", "emails", "draft"])
print("\nretrieved for 'draft an email':")
for m in hits:
    print(f"  #{m['id']}: {m['text']}")
# The stale 'formal tone' memory is tombstoned - only the correction returns.


## Practice

1. Add an `open_questions` field to `compact` and a turn that should land
   there. What happens downstream if it's dropped instead?
2. Remove the `superseded_by` filter and rerun retrieval — describe the bug
   the model would now exhibit when drafting the email.
3. Design the write policy: for each of five sample utterances, decide
   memory-worthy or not, and which type (semantic/episodic/procedural).
4. Your agent's context is 85% full mid-task with an active tool loop. Walk
   through exactly what you keep, compact, and evict — and why compacting
   the tool results last turn would be a mistake.

## Design Checklist

- [ ] Token usage tracked per context section; compaction triggers at ~70-80%, not at overflow.
- [ ] Compaction is structured; decisions, constraints, identifiers survive verbatim.
- [ ] Long-term memory retrieved by relevance x recency, top-k, labeled in context.
- [ ] Corrections tombstone superseded entries.
- [ ] Explicit memory write policy; writes logged with reasons.
- [ ] Memory scoped per user/tenant; TTLs and deletion supported.